tanh() is used as activation function in NER Whh is weight at recurrent neuron.

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
from datasets import load_dataset
from collections import Counter
import numpy as np

In [ ]:
# 1. Load dataset and build vocab
dataset = load_dataset("imdb")

train_texts = dataset['train']['text']
train_labels = dataset['train']['label']

test_texts = dataset['test']['text']
test_labels = dataset['test']['label']

def build_vocab(texts, max_size=10000):
    words = []
    for text in texts:
        words.extend(text.lower().split())
    freq = Counter(words)
    vocab = {word: i+1 for i, (word, _) in enumerate(freq.most_common(max_size))}
    return vocab

vocab = build_vocab(train_texts)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

plain_text/train-00000-of-00001.parquet:   0%|          | 0.00/21.0M [00:00<?, ?B/s]

plain_text/test-00000-of-00001.parquet:   0%|          | 0.00/20.5M [00:00<?, ?B/s]

plain_text/unsupervised-00000-of-00001.p(…):   0%|          | 0.00/42.0M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating unsupervised split:   0%|          | 0/50000 [00:00<?, ? examples/s]

In [ ]:
# 2. Encode texts to fixed-length sequences
def encode(text, vocab, max_len=100):
    tokens = text.lower().split()
    idxs = [vocab.get(token, 0) for token in tokens]  # 0 for unknown words
    if len(idxs) < max_len:
        idxs += [0] * (max_len - len(idxs))
    else:
        idxs = idxs[:max_len]
    return idxs

In [ ]:
# 3. Custom dataset class
class TextDataset(Dataset):
    def __init__(self, texts, labels, vocab):
        self.texts = texts
        self.labels = labels
        self.vocab = vocab

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        x = torch.tensor(encode(self.texts[idx], self.vocab), dtype=torch.long)
        y = torch.tensor(self.labels[idx], dtype=torch.long)
        return x, y

train_dataset = TextDataset(train_texts, train_labels, vocab)
test_dataset = TextDataset(test_texts, test_labels, vocab)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=64)

In [ ]:
# 4. Define model
class RNNClassifier(nn.Module):
    def __init__(self, vocab_size, embed_dim=50, hidden_dim=64, output_dim=2):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size + 1, embed_dim, padding_idx=0)
        self.rnn = nn.RNN(embed_dim, hidden_dim, batch_first=True)
        self.dropout = nn.Dropout(0.5)
        self.fc = nn.Linear(hidden_dim, output_dim)

    def forward(self, x):
        x = self.embedding(x)
        out, _ = self.rnn(x)
        out = self.dropout(out[:, -1, :])  # apply dropout on last hidden state
        out = self.fc(out)
        return out


model = RNNClassifier(vocab_size=len(vocab))

In [ ]:
# 5. Loss and optimizer
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

In [ ]:
# 6. Training loop
def train_epoch(model, loader, optimizer, criterion, device):
    model.train()
    total_loss = 0
    for x_batch, y_batch in loader:
        x_batch, y_batch = x_batch.to(device), y_batch.to(device)

        optimizer.zero_grad()
        outputs = model(x_batch)
        loss = criterion(outputs, y_batch)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
    return total_loss / len(loader)

In [ ]:
# 7. Evaluation function
def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss = 0
    correct = 0
    total = 0
    with torch.no_grad():
        for x_batch, y_batch in loader:
            x_batch, y_batch = x_batch.to(device), y_batch.to(device)
            outputs = model(x_batch)
            loss = criterion(outputs, y_batch)
            total_loss += loss.item()

            preds = outputs.argmax(dim=1)
            correct += (preds == y_batch).sum().item()
            total += y_batch.size(0)
    return total_loss / len(loader), correct / total

In [ ]:
# 8. Run training
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

epochs = 50
for epoch in range(epochs):
    train_loss = train_epoch(model, train_loader, optimizer, criterion, device)
    val_loss, val_acc = evaluate(model, test_loader, criterion, device)
    print(f"Epoch {epoch+1}: Train loss={train_loss:.4f}, Val loss={val_loss:.4f}, Val acc={val_acc:.4f}")

Epoch 1: Train loss=0.7009, Val loss=0.6932, Val acc=0.5129
Epoch 2: Train loss=0.6925, Val loss=0.6931, Val acc=0.5053
Epoch 3: Train loss=0.6933, Val loss=0.6935, Val acc=0.5011
Epoch 4: Train loss=0.6944, Val loss=0.6962, Val acc=0.5046
Epoch 5: Train loss=0.6911, Val loss=0.6906, Val acc=0.5304
Epoch 6: Train loss=0.6876, Val loss=0.6927, Val acc=0.5239
Epoch 7: Train loss=0.6861, Val loss=0.6918, Val acc=0.5322
Epoch 8: Train loss=0.6754, Val loss=0.6914, Val acc=0.5251
Epoch 9: Train loss=0.6799, Val loss=0.6891, Val acc=0.5585
Epoch 10: Train loss=0.6557, Val loss=0.6790, Val acc=0.5668
Epoch 11: Train loss=0.6560, Val loss=0.6878, Val acc=0.5714
Epoch 12: Train loss=0.6377, Val loss=0.7016, Val acc=0.5452
Epoch 13: Train loss=0.6196, Val loss=0.6621, Val acc=0.6273
Epoch 14: Train loss=0.5976, Val loss=0.7002, Val acc=0.5260
Epoch 15: Train loss=0.6363, Val loss=0.7633, Val acc=0.5729
Epoch 16: Train loss=0.6275, Val loss=0.7249, Val acc=0.5499
Epoch 17: Train loss=0.5948, Val 

In [ ]:
test_acc = evaluate(model, test_loader, criterion, device)
print(f"Test Accuracy: {round(test_acc[1] * 100, 2)}%")

Test Accuracy: 53.42%
